# Analysis of CWEs by abstraction level

This notebook analyzes the CWE distribution by grouping weaknesses according to the `abstraction` field.

For each level, it computes:

- **CWE count**
- **Median CVEs**
- **Total associations**

The analysis helps identify which hierarchy levels contain the most categories and where most CVE-CWE associations are concentrated.


In [1]:
from pathlib import Path
import pandas as pd

PROJECT_FILE = Path("data/cwe_without_prohibited_with_counts.csv")
UPLOADED_FILE = Path("/mnt/data/Pasted text(2).txt")

if PROJECT_FILE.exists():
    input_file = PROJECT_FILE
elif UPLOADED_FILE.exists():
    input_file = UPLOADED_FILE
else:
    raise FileNotFoundError("CWE file not found.")

df = pd.read_csv(input_file)
print(f"File used: {input_file}")
print(f"Number of CWEs: {len(df)}")
df.head()


File used: data\cwe_without_prohibited_with_counts.csv
Number of CWEs: 718


,cwe_id,name,abstraction,status,mapping_usage,primary_parent,other_parents,children,depth,cve_count
0,CWE-5,J2EE Misconfiguration: Data Transmission Witho...,Variant,Draft,Allowed,CWE-319,[],[],3,2
1,CWE-6,J2EE Misconfiguration: Insufficient Session-ID...,Variant,Incomplete,Allowed,CWE-334,[],[],3,1
2,CWE-11,ASP.NET Misconfiguration: Creating Debug Binary,Variant,Draft,Allowed,CWE-489,[],[],2,2
3,CWE-12,ASP.NET Misconfiguration: Missing Custom Error...,Variant,Draft,Allowed,CWE-756,[],[],3,1
4,CWE-14,Compiler Removal of Code to Clear Buffers,Variant,Draft,Allowed,CWE-733,[],[],4,10


## Column check

This checks that `abstraction`, `cwe_id`, and `cve_count` are present.  
CVE counts are converted to numeric format before aggregation.


In [2]:
required_columns = {"abstraction", "cwe_id", "cve_count"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Colonne mancanti: {sorted(missing_columns)}")

df["abstraction"] = df["abstraction"].fillna("Unknown").str.strip()
df["cve_count"] = pd.to_numeric(df["cve_count"], errors="raise")


## Summary table

The table groups CWEs by abstraction level and computes:

- how many CWEs belong to each level;
- the median number of associated CVEs;
- la somma complessiva delle associazioni CVE–CWE.


In [3]:
summary = (
    df.groupby("abstraction", dropna=False)
    .agg(
        **{
            "CWE count": ("cwe_id", "nunique"),
            "Median CVEs": ("cve_count", "median"),
            "Total associations": ("cve_count", "sum"),
        }
    )
    .reset_index()
    .rename(columns={"abstraction": "Abstraction"})
    .sort_values("CWE count", ascending=False)
    .reset_index(drop=True)
)

summary


,Abstraction,CWE count,Median CVEs,Total associations
0,Base,400,17.5,184019
1,Variant,203,5.0,23533
2,Class,98,66.0,86960
3,Pillar,10,140.5,6833
4,Compound,7,105.0,10128


## Column meanings

### Abstraction

Indicates the CWE abstraction level in the MITRE hierarchy:

- **Pillar**: very general concepts;
- **Class**: broad families of weaknesses;
- **Base**: concrete and fairly specific weaknesses;
- **Variant**: very specific forms, often tied to technologies or contexts;
- **Compound**: weaknesses composed of multiple conditions or mechanisms.

### CWE count

Indicates how many CWE categories belong to each level.  
This helps describe how the catalog is distributed.

### Median CVEs

This is the median number of CVEs associated with the CWEs in the group.

The median is preferable to the mean because some classes have tens of thousands of examples and would strongly distort the average value.

A median equal to 5 means that half of the CWEs in the group have at most 5 CVEs and half have at least 5.

### Total associations

This is the sum of `cve_count` values for all CWEs at the same level.

It does not necessarily match the number of unique CVEs, because the same CVE can be associated with multiple CWEs. It therefore measures the total number of CVE-CWE associations.


## Interpretation for class balancing

The comparison between **CWE count** and **Median CVEs** highlights the long tail of the dataset.

A level may contain many categories while having a very low median. In this case, many rare classes exist.

**Total associations** instead shows where most of the data is concentrated.

This analysis motivates bottom-up aggregation:

1. classes above the threshold remain independent;
2. rare classes are progressively merged into their parent;
3. `Class` and `Pillar` nodes can also become final classes if they reach the minimum threshold.


In [4]:
OUTPUT_FILE = Path("data/cwe_abstraction_summary.csv")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_FILE, index=False)

print(f"Table saved to: {OUTPUT_FILE}")
summary


Table saved to: data\cwe_abstraction_summary.csv


,Abstraction,CWE count,Median CVEs,Total associations
0,Base,400,17.5,184019
1,Variant,203,5.0,23533
2,Class,98,66.0,86960
3,Pillar,10,140.5,6833
4,Compound,7,105.0,10128
